# 07 · Bounds, constraints, and slip directions

Regularization (Chapter 04) can make slip smooth or small, but it cannot say
"this thrust fault does not slip in reverse" or "coupling can never exceed one."
Those are *hard* limits: some models are simply not allowed, no matter how well
they fit. This chapter shows three ways to build such limits into an inversion,
non-negativity, upper bounds, and fixed slip directions, and it is honest about
the bias they can introduce when the assumption behind them is wrong.

**Learning objectives**

By the end of the chapter, you will be able to:

- distinguish a soft quadratic penalty from a hard constraint
- apply non-negativity and two-sided bounds with `geodef.solve`
- fix the slip direction with a rake or azimuth basis and halve the unknowns
- recognize the artifacts that hard constraints can create

**Prerequisites:** Chapter 04; Chapter 05 recommended
**Estimated time:** 60–90 minutes

GeoDef's axes, signs, and units are summarized in
[the conventions guide](../docs/conventions.md). New terms are introduced in
words here and collected in [the glossary](../docs/glossary.md).

## 1. Hard limits that regularization cannot express

Regularization scores every model on a sliding scale: a rough model is
*disliked*, but never *forbidden*. Enough data pull can always overcome the
penalty. Some geological knowledge is not like that. It draws a hard line:

- slip should not reverse its sense (no back-slip on a thrust): $\mathbf{m} \ge 0$;
- slip magnitude cannot exceed some physical cap: $\mathbf{m} \le u$;
- more general limits written as inequalities: $C\mathbf{m} \le \mathbf{d}$.

These are **constraints**. A constrained inversion searches only among the
models that satisfy them, and ignores the rest entirely, however well they fit.

## 2. How GeoDef applies constraints

`geodef.solve` reads two arguments and picks the right solver automatically:

- `bounds=(0, None)` forbids negative slip, giving **non-negative least
  squares** (NNLS);
- `bounds=(lower, upper)` keeps every value inside a box, giving **bounded
  least squares**;
- `constraints=(C, d)` with `method="constrained"` enforces the general
  inequality $C\mathbf{m} \le \mathbf{d}$, solved as a **quadratic program**.

Bounds may be a single pair applied to everything, one pair per component, or a
full array with its own limit for each parameter. You state the physics; the
solver enforces it exactly.

## 3. Rebuild the shared scenario

We reuse the megathrust, true slip, GNSS network, and 1 cm noise from
Chapter 03. The setup cell is the familiar one.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

%matplotlib inline

rng = np.random.default_rng(0)

In [ ]:
# The shared megathrust, true slip, GNSS network, and 1 cm noise.
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,
    strike=315.0, dip=15.0,
    length=180e3, width=90e3,
    n_length=12, n_width=6,
)
N = fault.n_patches
n_length, n_width = fault.grid_shape
along = np.arange(N) % n_length
down = np.arange(N) // n_length
bump = np.exp(-(((along - (n_length - 1) / 2) / 3.0) ** 2
                + ((down - n_width / 2) / 1.5) ** 2))
slip_true = np.concatenate([np.zeros(N), 3.0 * bump])

grid_lon, grid_lat = np.meshgrid(np.linspace(98.5, 101.5, 8),
                                 np.linspace(-3.6, -0.4, 8))
station_lon, station_lat = grid_lon.ravel(), grid_lat.ravel()
n_stations = station_lon.size

G_full = fault.greens_matrix(station_lat, station_lon)
sigma = 0.01
d_obs = G_full @ slip_true + rng.normal(0.0, sigma, 3 * n_stations)
gnss = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=d_obs[0::3], north=d_obs[1::3], up=d_obs[2::3],
    sigma_east=np.full(n_stations, sigma),
    sigma_north=np.full(n_stations, sigma),
    sigma_up=np.full(n_stations, sigma),
)
print(f"{N} patches, {n_stations} stations")

## 4. Non-negative slip

A lightly smoothed unconstrained inversion dips slightly *negative* near the
fault edges: a little spurious back-slip, the leftover noise amplification of
Chapter 03. Since our true slip is one-signed thrust, negative slip is
physically wrong, and requiring $\mathbf{m} \ge 0$ removes it. Passing
`bounds=(0, None)` switches `geodef.solve` to NNLS automatically.

In [ ]:
settings = dict(components="dip", regularization="laplacian",
                regularization_strength=0.2)
free = geodef.solve(fault, gnss, **settings)                 # can go negative
non_negative = geodef.solve(fault, gnss, bounds=(0, None), **settings)

print(f"unconstrained minimum slip: {free.dip_slip.min() * 100:+.1f} cm "
      f"(spurious back-slip)")
print(f"non-negative minimum slip:  {non_negative.dip_slip.min() * 100:+.1f} cm")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
limit = abs(free.dip_slip).max()
geodef.plot.slip(fault, free.dip_slip, ax=axes[0], cmap="RdBu_r",
                 vmin=-limit, vmax=limit, title="Unconstrained",
                 colorbar_label="Slip (m)")
geodef.plot.slip(fault, non_negative.dip_slip, ax=axes[1], cmap="RdBu_r",
                 vmin=-limit, vmax=limit, title="Non-negative (NNLS)",
                 colorbar_label="Slip (m)")
plt.tight_layout()

The back-slip is gone. This is a case where the constraint is genuinely
correct, so it improves the result at no cost. That will not always be true, as
the next section shows.

## 5. Bounded slip, and the bias a wrong bound creates

Bounds can be two-sided. Suppose we (incorrectly) believed slip could not
exceed 1 m and imposed that as an upper cap. The true peak is nearly 3 m, so
this bound is wrong, and watching it "bite" is instructive.

In [ ]:
capped = geodef.solve(fault, gnss, bounds=(0, 1.0), **settings)
print(f"peak slip, uncapped: {non_negative.dip_slip.max():.2f} m")
print(f"peak slip, capped:   {capped.dip_slip.max():.2f} m (pinned at the 1 m ceiling)")
print(f"reduced chi^2: uncapped {non_negative.reduced_chi2:.2f}, "
      f"capped {capped.reduced_chi2:.2f}")

The capped solution slams against the ceiling, and its reduced chi-squared
climbs: forced away from the slip the data actually want, it now fits the data
worse. This is the general lesson about constraints. When the assumption is
right (no back-slip), it helps for free. When it is wrong (a too-low cap), it
biases the answer and degrades the fit. The fit statistic is your warning
light: a constraint that noticeably worsens chi-squared is fighting the data.

## 6. Fixing the slip direction

A different kind of prior fixes the *direction* of slip and solves for a single
amplitude per patch, rather than two independent components. This encodes a
sense of motion cleanly and halves the number of unknowns.

- `components="rake"` fixes one rake angle in each patch's local strike-dip
  frame. A rake of 90 degrees is pure dip slip (reverse), which matches our
  true model.
- `components="azimuth"` fixes a geographic slip direction, which stays
  consistent even when the fault strike varies from patch to patch.

Because the reduction is exact (each amplitude maps to fixed strike and dip
components), it is a hard assumption, not a soft preference.

In [ ]:
rake = geodef.solve(fault, gnss, components="rake", rake=90.0,
                    regularization="laplacian", regularization_strength=1.0)
print(f"components='rake' solves {rake.dip_slip.size} amplitudes "
      f"(versus {2 * N} for a full two-component solve)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
vmax = slip_true[N:].max()
geodef.plot.slip(fault, slip_true[N:], ax=axes[0], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, title="True", colorbar_label="Slip (m)")
geodef.plot.slip(fault, rake.dip_slip, ax=axes[1], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, title="Fixed-rake amplitude",
                 colorbar_label="Slip (m)")
plt.tight_layout()

With the direction fixed correctly, the fixed-rake solve recovers the asperity
using half the parameters. The saving is real, but it rests on knowing the rake
in advance; a wrong rake would bias the result just as a wrong bound did.

**Checkpoint.** Non-negativity often produces small isolated spots of slip
pinned right at the fault edges. Why does forbidding negative slip tend to push
amplitude toward the edges?

<details><summary>Answer</summary>

Without the constraint, the inversion balances a positive patch against a small
negative neighbor. Forbidding negatives removes that balancing move, so the
optimizer instead parks leftover amplitude on edge patches, which the data
barely constrain and which therefore cost little to adjust. The result is a
constraint-induced artifact, not a real feature.

</details>

## Checkpoint questions

**What is the difference between a bound and a regularization penalty?**

<details><summary>Answer</summary>

A bound *excludes* models: anything outside the allowed set is off the table
regardless of fit. A regularization penalty *ranks* all models on a continuous
scale, and a strong enough data pull can always overcome it. One is a wall; the
other is a slope.

</details>

**When should you prefer an azimuth basis over a rake basis?**

<details><summary>Answer</summary>

When the fault strike varies from patch to patch. A single rake is defined in
each patch's local frame, so the same rake can point in different geographic
directions along a curved fault. Fixing a geographic azimuth keeps the slip
direction consistent across all patches.

</details>

**How can you tell that a constraint is biasing your solution?**

<details><summary>Answer</summary>

Watch the fit. A correct constraint leaves the reduced chi-squared essentially
unchanged; a constraint that fights the data forces a visibly worse fit, and
often piles slip against the constraint boundary. Report which parameters sit
at their bounds along with the solution.

</details>

## Common mistakes

- **Constraining the wrong signed component.** Applying non-negativity to a
  component whose positive direction is defined the opposite way enforces the
  reverse mechanism. Check the sign convention before adding a bound.
- **Treating a pinned parameter as Gaussian.** A parameter resting exactly at a
  bound has a one-sided, asymmetric uncertainty; the usual symmetric error bar
  does not apply to it.
- **One rake on a curved fault.** A single rake across a fault whose strike
  changes implies different geographic slip directions on different patches.
  Use an azimuth basis when strike varies.

## Recap

- Constraints encode hard limits (non-negativity, caps, inequalities) that
  regularization's soft penalties cannot express.
- `geodef.solve` selects NNLS, bounded least squares, or a quadratic program
  from the `bounds` and `constraints` arguments.
- Fixing the slip direction with a rake or azimuth basis halves the unknowns
  and enforces a sense of motion exactly.
- A correct constraint helps for free; a wrong one biases the estimate and
  worsens the fit, so always check chi-squared and which parameters sit at
  their bounds.

Chapter 08 asks the complementary question: given a solution, how well do the
data actually resolve it, and how uncertain is it?

## Exercises

Predict the qualitative outcome before running each experiment. Worked
solutions are in `solutions/07_bounds_and_constraints_solutions.ipynb`.

1. **Three solvers.** Compare unconstrained weighted least squares, NNLS, and a
   fixed-rake solve on the same data. Tabulate their peak slip, reduced
   chi-squared, and any negative slip.
2. **Tighten the cap.** Lower the upper bound step by step and find the value at
   which the reduced chi-squared starts to rise substantially. What does that
   threshold tell you?
3. **Per-component bounds.** Run a two-component solve with different bounds on
   strike slip and dip slip, and describe the effect.
4. **Challenge: build the reduction by hand.** Construct the fixed-rake
   reduction that maps one amplitude per patch to strike and dip components,
   apply it to the Green's matrix, and confirm your prediction matches
   `components="rake"`.

## Further reading

- Lawson, C. L., and Hanson, R. J. (1995), *Solving Least Squares Problems*, on
  non-negative least squares.
- Nocedal, J., and Wright, S. J. (2006), *Numerical Optimization*, on bounded
  and constrained optimization.
- Segall, P. (2010), *Earthquake and Volcano Deformation*, on physically
  constrained geodetic inversion.
- [GeoDef inversion documentation](../docs/invert.md) for the `bounds`,
  `constraints`, and component-basis interfaces.
- The next course chapter is `08_uncertainty_and_resolution.ipynb`.